### 1. Fundamental Parameters

In [1]:
from model import alpha, self_energy
from smatrix import create_self_energy_interpolator_numba
import numpy as np
from eigenstate_solving import BZ_proj
from model import square_lattice
from W_state import q_bounds
from scipy.optimize import brentq


sigma_data = np.load("../../data/sigma_grid0f1a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)
collective_lamb_shift = self_energy(
    0, 0, square_lattice.a, square_lattice.d, square_lattice.omega_e, alpha
).real
sigma_func_period = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)
# r_para = np.array([-74, 30])
# p_para = np.array([60, 50])
r_para = np.array([5,0])
p_para = np.array([2, 0])

E = 2 * (square_lattice.omega_e + collective_lamb_shift)
E1 = (E-1) / 2

Q_para = np.array([0, 0])
Zc = 0

if np.linalg.norm(r_para) + np.linalg.norm(BZ_proj(Q_para-r_para,square_lattice)) > E:
    raise ValueError("In plane-wave basis, the total in-plane photon momenta exceeds the total energy.")

if np.linalg.norm(p_para) + np.linalg.norm(BZ_proj(Q_para-p_para,square_lattice)) > E:
    raise ValueError("In W-state label, the total in-plane photon momenta exceeds the total energy.")

n_points = int(2e6) # number of grid for FFT

q_min, q_max = q_bounds(E, r_para, Q_para, square_lattice)
q_grid = np.linspace(q_min + 1e-10, q_max - 1e-10, n_points, endpoint=False)
dq = q_grid[1] - q_grid[0]

"""
k_para = np.array([50, 50])
E = square_lattice.omega_e

kz = np.sqrt(E**2 - np.hypot(k_para[0], k_para[1]) ** 2)
k = np.concatenate((k_para, np.array([kz])))

print(square_lattice.ge(k))
print(
    (
        -2
        * square_lattice.a**2
        * np.imag(sigma_func_period(k_para[0], k_para[1]))
        * (kz / E)
    )
    ** (1 / 2)
)
"""


'\nk_para = np.array([50, 50])\nE = square_lattice.omega_e\n\nkz = np.sqrt(E**2 - np.hypot(k_para[0], k_para[1]) ** 2)\nk = np.concatenate((k_para, np.array([kz])))\n\nprint(square_lattice.ge(k))\nprint(\n    (\n        -2\n        * square_lattice.a**2\n        * np.imag(sigma_func_period(k_para[0], k_para[1]))\n        * (kz / E)\n    )\n    ** (1 / 2)\n)\n'

Near the zeros of the denominator, its Taylor expansion is $\partial_q \Big[(\tilde{E}_1 -\omega_e -\Sigma(\mathbf{r}_\parallel))(E-\tilde{E}_1-\omega_e-\Sigma(r(Q_\parallel-r_\parallel))\left(t(r_\parallel,\tilde{E}_1)t(r(Q_\parallel-r_\parallel),E-\tilde{E}_1)\right)|_{q=\mathrm{root}}\times (E-\tilde{E}_1)\Big]+i\eta$ where $\tilde{E}_1=\sqrt{|k_\parallel|^2+(K_z/2+q)^2}$. For the simplicity of notation, we define $D_0(\mathbf{r}_\parallel;E_1,\mathbf{p}_\parallel,E,\mathbf{Q}_\parallel):=\partial_q \left(t(r_\parallel,\tilde{E}_1)t(r(Q_\parallel-r_\parallel),E-\tilde{E}_1)\right)|_{q=\mathrm{root}}$ We can factor out the derivative term, obtaining $D_0(\mathbf{r}_\parallel;E_1,\mathbf{p}_\parallel,E,\mathbf{Q}_\parallel)\times \Big[q-q_{\mathrm{root}}+i\eta/\left(D_0(\mathbf{r}_\parallel;E_1,\mathbf{p}_\parallel,E,\mathbf{Q}_\parallel)\right) \Big]$. We see that the second term is a Lorentzian function with a peak width $2\eta/\operatorname{Re}\Big[D_0(\mathbf{r}_\parallel;E_1,\mathbf{p}_\parallel,E,\mathbf{Q}_\parallel)\Big]$. To guarentee that this peak the resolved by `n_Rs` points in q grid. $\eta$ is estimated as $\eta=1/2\,R_s\,\Delta q\,D_0(\mathbf{r}_\parallel;E_1,\mathbf{p}_\parallel,E,\mathbf{Q}_\parallel)$

In [2]:
from scipy.differentiate import derivative
from W_state import _denom_BM
from smatrix import t_reg

def _pole_loc(r_para, p_para, E1, E, Q_para,eta, lattice, sigma_func_period):
        """Find pole locations along the q axis using the quadratic formula.

        Returns
        -------
        list[list[float]]
            A list of pole locations. Each entry is ``[q, Kz]``, where ``q`` is the
            real pole coordinate along the q axis and ``Kz`` is the corresponding
            total z-direction wavevector. Small numerical imaginary parts are
            discarded before returning.
        """
        s_para = BZ_proj(Q_para - r_para, lattice)
        # eigenvalue of this W state
        tt = t_reg(p_para, E1, lattice, sigma_func_period) * t_reg(
            BZ_proj(Q_para - p_para, lattice), E - E1, lattice, sigma_func_period
        )
        # self-energy terms
        Sigma1 = sigma_func_period(r_para[0], r_para[1])
        Sigma2 = sigma_func_period(s_para[0], s_para[1])

        # the follows are the coefficient terms in the quadratic equation
        # coefficient for E1tilde**2
        A = tt - 1
        # coefficient for E1tilde
        B = (
            E * (1 - tt)
            + (
                sigma_func_period(s_para[0], s_para[1])
                - sigma_func_period(r_para[0], r_para[1])
            )
            * tt
            + np.conj(sigma_func_period(r_para[0], r_para[1]))
            - np.conj(sigma_func_period(s_para[0], s_para[1]))
        )
        # constant term
        C = (
            E * Sigma1 * tt
            - Sigma1 * Sigma2 * tt
            - E * lattice.omega_e
            + E * tt * lattice.omega_e
            - Sigma1 * tt * lattice.omega_e
            - Sigma2 * tt * lattice.omega_e
            + lattice.omega_e**2
            - tt * lattice.omega_e**2
            - E * np.conjugate(Sigma1)
            + lattice.omega_e * np.conjugate(Sigma1)
            + lattice.omega_e * np.conjugate(Sigma2)
            + np.conjugate(Sigma1) * np.conjugate(Sigma2)
        )

        C = C+1j*eta

        discriminant = B**2 - 4 * A * C
        discriminant = complex(discriminant)

        # quadratic formula
        root = [
            (-B + np.sqrt(discriminant)) / (2 * A),
            (-B - np.sqrt(discriminant)) / (2 * A),
        ]

        # The imaginary part of the root for E1tilde never vanishes exactly due to numerical errors, but it should be discarded when it is small enough.
        if np.any(np.abs(np.imag(root)) > 1e-6):
            return None

        val_list = []
        # convert the root in E1tilde to the corresponding q and Kz values
        for E1tilde in root:
            rz = np.sqrt(E1tilde**2 - np.hypot(r_para[0], r_para[1]) ** 2)
            sz = np.sqrt((E - E1tilde) ** 2 - np.hypot(s_para[0], s_para[1]) ** 2)

            q = (rz - sz) / 2
            Kz = rz + sz
            if np.abs(np.imag(q)) > 1e-5:
                return None
            if np.abs(np.imag(Kz)) > 1e-5:
                return None

            # discard the small imainary part of q and Kz
            q = np.real(q)
            
            q_min, q_max = q_bounds(E, r_para, Q_para, square_lattice)
            if q < q_min or q > q_max:
                continue
            Kz = np.real(Kz)
            val_list.append([q, Kz])

        return val_list

def eta_estimator(r_para, p_para, E1, E, Q_para, lattice, sigma_func_period, dq, n_Rs):
    """Estimate the imaginary regulator needed to resolve q-axis poles.

    The denominator can develop real poles as a function of the relative
    z-momentum ``q``. This routine locates those candidate poles,
    discards poles with negligible imaginary parts or poles outside the
    q-grid bounds, and returns ``0.0`` when no regulator is needed.

    For real poles inside the grid, ``eta`` is chosen from the largest local
    slope of ``Re(denominator)`` at the pole locations. The estimate broadens
    the Lorentzian peak so its width is resolved by approximately ``n_Rs``
    grid spacings of size ``dq``.
    """

    q_root_list = _pole_loc(
        r_para, p_para, E1, E, Q_para, lattice, sigma_func_period
    )

    if (
        q_root_list is None
    ):  # if the pole is not on the real axis, we do not need the regulator eta.
        print("Pole is not on the real axis, no regulator needed. eta=0.0")
        return 0.0
    elif not q_root_list:  # if the pole is on the real axis but outside of the q grid, we also do not need the regulator eta.
        print(f"Pole is not in the interval of [{q_min}, {q_max}], no regulator needed. eta=0.0")
        return 0.0
    else:
        q_roots = [root[0] for root in q_root_list]

        def denom_complex_scalar(q):
            return _denom_BM(
                r_para,
                float(q),
                p_para,
                E1,
                E,
                Q_para,
                lattice,
                sigma_func_period,
            )

        def denom_re(q):
            return np.vectorize(
                lambda x: np.real(denom_complex_scalar(x)), otypes=[float]
            )(q)

        d_re_values = [np.abs(derivative(denom_re, q_root).df) for q_root in q_roots]
        d_re_max = max(d_re_values)

        estimated_eta = 0.5 * n_Rs * d_re_max * dq
    #    print(f"Estimated eta for resolving the pole: {estimated_eta}")

        return estimated_eta


In [7]:
def peak_width_estimator(r_para, p_para, E1, E, Q_para,eta,square_lattice, sigma_func_period_numba, xatol=1e-10):
    from scipy.optimize import minimize_scalar

    root_list = _pole_loc(r_para, p_para, E1, E, Q_para, 0.0,square_lattice, sigma_func_period_numba)


    if not root_list:
        print("No pole on the real axis, cannot estimate peak width.")
        return None
    q_root_list = sorted(root[0] for root in root_list)

    if len(q_root_list) == 2 and np.isclose(q_root_list[0], q_root_list[1]):
        q_root_list = [(q_root_list[0] + q_root_list[1]) / 2]

    def denom_abs_profile(q):
        return np.abs(_denom_BM(r_para, q, p_para, E1, E, Q_para, eta,square_lattice, sigma_func_period_numba)[1])



    peak_width_list = []

    if len(q_root_list) == 1:
        # when there is a root for the unregularised denominator, eta might push the pole off the real axis. 
        # In this case, we need to re-estimate the peak location by finding the minimum of the absolute value of the denominator, and then calculate the peak width based on this estimated peak location.
        min_val = minimize_scalar(denom_abs_profile, bounds=(q_min, q_max), method="bounded", options={"xatol": xatol})
        peak_val = 1 / min_val.fun
        peak_loc = min_val.x
        threshold =  peak_val / 2
        q_L = brentq(lambda q: denom_abs_profile(q) - 1/threshold, q_min, peak_loc, xtol=xatol, rtol=max(xatol, 1e-15))
        q_R = brentq(lambda q: denom_abs_profile(q) - 1/threshold, peak_loc, q_max, xtol=xatol, rtol=max(xatol, 1e-15))
        peak_width = q_R - q_L
        peak_width_list.append(peak_width)

    if len(q_root_list) == 2:
        # In the regularised case with two roots, the new poles is still close to the original poles. So we use the midpoint of two original roots to split the regions for searching the left and right peaks.
        mid_point = (q_root_list[0] + q_root_list[1]) / 2
        min_val_L = minimize_scalar(denom_abs_profile, bounds=(q_min, mid_point), method="bounded", options={"xatol": xatol})
        min_val_R = minimize_scalar(denom_abs_profile, bounds=(mid_point, q_max), method="bounded", options={"xatol": xatol})

        # calculate the peak width for the left peak
        peak_val_L = 1 / min_val_L.fun
        peak_loc_L = min_val_L.x
        threshold_L = peak_val_L / 2
        peak_val_R = 1 / min_val_R.fun
        peak_loc_R = min_val_R.x
        threshold_R = peak_val_R / 2

        q_L1 = brentq(lambda q: denom_abs_profile(q) - 1/threshold_L, q_min, peak_loc_L, xtol=xatol, rtol=max(xatol, 1e-15))
        q_R1 = brentq(lambda q: denom_abs_profile(q) - 1/threshold_L, peak_loc_L, mid_point, xtol=xatol, rtol=max(xatol, 1e-15))
        peak_width_list.append(q_R1 - q_L1)

        # calculate the peak width for the right peak
        peak_val = 1 / minimize_scalar(denom_abs_profile, bounds=(mid_point, q_max), method="bounded", options={"xatol": xatol}).fun
        threshold = peak_val / 2
        q_L2 = brentq(lambda q: denom_abs_profile(q) - 1/threshold_R, mid_point, peak_loc_R, xtol=xatol, rtol=max(xatol, 1e-15))
        q_R2 = brentq(lambda q: denom_abs_profile(q) - 1/threshold_R, peak_loc_R, q_max, xtol=xatol, rtol=max(xatol, 1e-15))
        peak_width_list.append(q_R2 - q_L2)

    width_min = min(peak_width_list)
    return width_min

def peak_width_estimator_fixed(
    r_para,
    p_para,
    E1,
    E,
    Q_para,
    eta,
    square_lattice,
    sigma_func_period_numba,
    half_height_fraction=0.5,
    scan_points=2000,
    crossing_scan_points=2000,
):
    from scipy.optimize import brentq, minimize_scalar

    if not 0 < half_height_fraction < 1:
        raise ValueError("half_height_fraction must be between 0 and 1.")

    q_min_local, q_max_local = q_bounds(E, r_para, Q_para, square_lattice)
    q_span = q_max_local - q_min_local
    edge_eps = max(1e-12, q_span * 1e-12)
    q_min_search = q_min_local + edge_eps
    q_max_search = q_max_local - edge_eps
    if q_min_search >= q_max_search:
        print("q interval is too small to estimate peak width.")
        return None

    def denom_abs_profile(q):
        return np.abs(
            _denom_BM(
                r_para,
                float(q),
                p_para,
                E1,
                E,
                Q_para,
                eta,
                square_lattice,
                sigma_func_period_numba,
            )[1]
        )

    # Large eta can shift or merge peaks, so find real-axis peak candidates
    # from local minima of |D(q) + i eta| rather than from _pole_loc.
    scan_q = np.linspace(q_min_search, q_max_search, scan_points)
    scan_denom = np.array([denom_abs_profile(q) for q in scan_q])
    finite_mask = np.isfinite(scan_denom)
    if not np.any(finite_mask):
        print("No finite denominator values found on the q interval.")
        return None

    candidate_idx = {int(np.nanargmin(scan_denom))}
    for i in range(1, len(scan_q) - 1):
        if not (finite_mask[i - 1] and finite_mask[i] and finite_mask[i + 1]):
            continue
        if scan_denom[i] <= scan_denom[i - 1] and scan_denom[i] <= scan_denom[i + 1]:
            candidate_idx.add(i)

    peak_candidates = []
    for i in sorted(candidate_idx):
        left = scan_q[max(i - 1, 0)]
        right = scan_q[min(i + 1, len(scan_q) - 1)]
        if left == right:
            continue
        min_val = minimize_scalar(
            denom_abs_profile,
            bounds=(left, right),
            method="bounded",
            options={"xatol": max(1e-14, (right - left) * 1e-12)},
        )
        if min_val.success and np.isfinite(min_val.fun):
            peak_candidates.append((float(min_val.x), float(min_val.fun)))

    if not peak_candidates:
        print("No eta-regularized peak found on the real q axis.")
        return None

    peak_candidates = sorted(peak_candidates, key=lambda item: item[0])
    merge_distance = 2 * q_span / max(scan_points - 1, 1)
    merged_peaks = []
    for peak_loc, denom_min in peak_candidates:
        if merged_peaks and abs(peak_loc - merged_peaks[-1][0]) <= merge_distance:
            if denom_min < merged_peaks[-1][1]:
                merged_peaks[-1] = (peak_loc, denom_min)
        else:
            merged_peaks.append((peak_loc, denom_min))

    def nearest_crossing(peak_loc, edge, denom_threshold):
        scan = np.linspace(peak_loc, edge, crossing_scan_points)
        values = np.array([denom_abs_profile(q) - denom_threshold for q in scan])
        if values[0] >= 0:
            return peak_loc
        for i in range(len(scan) - 1):
            if not (np.isfinite(values[i]) and np.isfinite(values[i + 1])):
                continue
            if values[i] == 0:
                return scan[i]
            if values[i] * values[i + 1] <= 0:
                return brentq(
                    lambda q: denom_abs_profile(q) - denom_threshold,
                    scan[i],
                    scan[i + 1],
                )
        print("Half-height crossing not found before the q-interval edge; using the edge as a bound.")
        return edge

    peak_width_list = []
    for peak_loc, denom_min in merged_peaks:
        # Half-height for 1 / |D| means |D| = |D|_min / half_height_fraction.
        denom_threshold = denom_min / half_height_fraction
        q_L = nearest_crossing(peak_loc, q_min_search, denom_threshold)
        q_R = nearest_crossing(peak_loc, q_max_search, denom_threshold)
        peak_width_list.append(q_R - q_L)

    return min(peak_width_list)


print(peak_width_estimator(r_para, p_para, E1, E, Q_para,1e-5,square_lattice, sigma_func_period_numba,xatol=1e-12))
print(peak_width_estimator_fixed(r_para, p_para, E1, E, Q_para,1e-5,square_lattice, sigma_func_period_numba))




6.765457243318451e-07
7.054826873797992e-07


In [24]:
from scipy.optimize import minimize_scalar
minimize_scalar(lambda x:np.abs(_denom_BM(r_para, x, p_para, E1, E, Q_para,0, square_lattice, sigma_func_period_numba)[1]),None,[q_min,q_max]).fun

np.float64(8.973191868313375e-09)

In [36]:
print(sorted([1,2]))

[1, 2]


### 2. Discrete FFT Part

For the discrete FFT grid used below, $q_j=q_{\min}+j\Delta q$ with `endpoint=False`, so $\Delta q\simeq(q_{\max}-q_{\min})/n$ (up to the small endpoint offset). The conjugate coordinate grid is $z_k=2\pi\,\mathrm{fftfreq}(n,d=\Delta q)$, so the non-aliased range is $|z|\le L$ with $L=\pi/\Delta q$, and the spacing is $\Delta z=2L/n=2\pi/(n\Delta q)$.

In [ ]:
from scipy.fft import ifft
from W_state import W_k_sp_grid
import matplotlib.pyplot as plt



eta_estimated = eta_estimator(
    r_para, p_para, E1, E, Q_para, square_lattice, sigma_func_period, dq, 10
)
print(f"Estimated eta for resolving the pole: {eta_estimated}")
eta = 0.0
q_grid, value_grid = W_k_sp_grid(
    r_para,
    p_para,
    E1,
    E,
    Q_para,
    Zc,
    square_lattice,
    sigma_func_period,
    n_points,
    q_grid,
    eta=eta,
)
L = np.pi / dq  # Nyquist limit

z_grid = 2 * np.pi * np.fft.fftfreq(n_points, d=dq)

# value_grid_ifft = (q_max-q_min)*ifft(value_grid)
value_grid_ifft = n_points * dq * ifft(value_grid)
value_grid_ifft *= np.exp(1j * q_grid[0] * z_grid)

# Only compare where the q grid resolves exp(i q z) with several samples per period.
nyquist_z = np.pi / dq
min_samples_per_period = 8
resolved_z = 2 * np.pi / (min_samples_per_period * dq)
keep = np.abs(z_grid) <= resolved_z


plot_idx = np.argsort(z_grid[keep])
plt.plot(z_grid[keep][plot_idx], np.real(value_grid_ifft[keep][plot_idx]))
plt.plot(z_grid[keep][plot_idx], np.imag(value_grid_ifft[keep][plot_idx]))
plt.show()

In [ ]:
from joblib import Parallel, delayed
from W_state import quad_FT

plot_mask = np.abs(z_grid) <= 20
plot_z_grid = z_grid[plot_mask]
plot_value_grid_ifft = value_grid_ifft[plot_mask]

n_intg_sample_point = 100
sample_step = max(1, len(plot_z_grid) // n_intg_sample_point)
z_sample = plot_z_grid[::sample_step]

n_quad_jobs = 6


def _compute_quad_ft(z_val):
    return quad_FT(
        r_para,
        p_para,
        Zc,
        z_val,
        E1,
        E,
        Q_para,
        square_lattice,
        sigma_func_period,
        eta=eta,
    )


quad_pairs = Parallel(n_jobs=n_quad_jobs, verbose=3)(
    delayed(_compute_quad_ft)(z_val) for z_val in z_sample
)
quad_result, quad_err = map(np.asarray, zip(*quad_pairs))


sorted_idx = np.argsort(plot_z_grid)
plt.plot(plot_z_grid[sorted_idx], np.real(plot_value_grid_ifft[sorted_idx]))
plt.plot(z_sample, np.real(quad_result), ".")
plt.show()

plt.plot(plot_z_grid[sorted_idx], np.imag(plot_value_grid_ifft[sorted_idx]))
plt.plot(z_sample, np.imag(quad_result), "v")
plt.show()

In [ ]:
mask_tmp = (q_grid <= 100) & (q_grid >= -100)
plt.plot(q_grid[mask_tmp], np.real(value_grid)[mask_tmp])
plt.plot(q_grid[mask_tmp], np.imag(value_grid)[mask_tmp])


In [ ]:
from collections import deque
from joblib import Parallel, delayed


r_para_path = deque()
n_r_grid_points = 5 # number of grid points along each direction in the r_para path inside the light cone

E = 2 * (square_lattice.omega_e + collective_lamb_shift)
r_grid_points =  np.linspace(0,E/2,n_r_grid_points,False)
r_path = deque()

# go through the grid in reverse order but skip the first point (0,0) to avoid duplication
for x in r_grid_points[:1:-1]:
    r_path.append(np.array([x, 0.0]))
"""
for x in r_grid_points:
    r_path.append([x, x])
"""

n_jobs = 6  # use all available CPU cores; set to a positive integer to limit workers

def _compute_q_distribution(r_para):
    q_grid, q_distribution =  W_k_sp_grid(
        r_para,
        p_para,
        E1,
        E,
        Q_para,
        Zc,
        square_lattice,
        sigma_func_period,
        n_points,
        eta=eta,
    )
    return q_grid, q_distribution

q_distribution_results = Parallel(n_jobs=n_jobs, verbose=3)(
    delayed(_compute_q_distribution)(r_para) for r_para in r_path
)
q_grid_list, q_distribution_list = map(list, zip(*q_distribution_results))





In [ ]:
r_heatmap_points = r_grid_points[:1:-1]
q_re = np.real(np.asarray(q_distribution_list))
q_im = np.imag(np.asarray(q_distribution_list))

if q_re.shape[0] != len(r_heatmap_points) or q_im.shape[0] != len(r_heatmap_points):
    raise ValueError("The number of q_distribution rows must match r_heatmap_points.")

def _plot_q_distribution_heatmap(values, colorbar_label, title):
    if "q_grid_list" in globals() and all(np.allclose(q_grid_list[0], q) for q in q_grid_list[1:]):
        fig, ax = plt.subplots(figsize=(8, 5))
        mesh = ax.pcolormesh(q_grid_list[0], r_heatmap_points, values, shading="auto")
    elif "q_grid_list" in globals():
        q_heatmap_grid = np.asarray(q_grid_list)
        r_heatmap_grid = np.repeat(r_heatmap_points[:, None], q_heatmap_grid.shape[1], axis=1)
        fig, ax = plt.subplots(figsize=(8, 5))
        mesh = ax.scatter(q_heatmap_grid.ravel(), r_heatmap_grid.ravel(), c=values.ravel(), s=8)
    else:
        fig, ax = plt.subplots(figsize=(8, 5))
        mesh = ax.pcolormesh(q_grid, r_heatmap_points, values, shading="auto")

    fig.colorbar(mesh, ax=ax, label=colorbar_label)
    ax.set_xlabel(r"$q$")
    ax.set_ylabel(r"$r_\parallel$")
    ax.set_title(title)
    plt.show()

_plot_q_distribution_heatmap(
    q_re,
    r"$\mathrm{Re}(q_{\mathrm{distribution}})$",
    r"$\mathrm{Re}(q_{\mathrm{distribution}})$ heatmap",
)
_plot_q_distribution_heatmap(
    q_im,
    r"$\mathrm{Im}(q_{\mathrm{distribution}})$",
    r"$\mathrm{Im}(q_{\mathrm{distribution}})$ heatmap",
)
